# 03c Transfer Evaluation

Evaluate Berlin → Leipzig transfer performance with comprehensive analysis.

**Inputs:**
- Berlin champions (from 03b)
- Leipzig test data (from 03a)
- Berlin evaluation results (from 03b)

**Outputs:**
- `transfer_evaluation.json` - Transfer metrics and analysis
- Transfer analysis figures (gap, robustness, hypotheses, etc.)

In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import pickle
import time
import datetime
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)

start_time = time.time()
config = load_experiment_config()
print(f"Transfer evaluation started")

In [ ]:
# Step 1: Load both ML + NN champions
print(f"\n{'='*60}")
print(f"STEP 1: Load Models")
print(f"{'='*60}")

model_dir = OUTPUT_DIR / 'models'

# Load ML champion
ml_model = training.load_model(model_dir / 'berlin_ml_champion.pkl')
ml_metadata = json.loads((model_dir / 'berlin_ml_champion.pkl.metadata.json').read_text())
ml_name = ml_metadata['model_name']

print(f"Loaded ML champion: {ml_name}")

# Load NN champion
nn_model = training.load_model(model_dir / 'berlin_nn_champion.pt')
nn_metadata = json.loads((model_dir / 'berlin_nn_champion.pt.metadata.json').read_text())
nn_name = nn_metadata['model_name']

print(f"Loaded NN champion: {nn_name}")

# Load label encoder and scaler
with (model_dir / 'label_encoder.pkl').open('rb') as f:
    label_encoder = pickle.load(f)
    label_to_idx = label_encoder['label_to_idx']
    idx_to_label = label_encoder['idx_to_label']

with (model_dir / 'berlin_scaler.pkl').open('rb') as f:
    berlin_scaler = pickle.load(f)

feature_cols = ml_metadata['feature_columns']
class_labels = list(idx_to_label.values())
print(f"\n{len(class_labels)} genera, {len(feature_cols)} features")

In [ ]:
# Load Leipzig test data
leipzig_finetune, leipzig_test = data_loading.load_leipzig_splits(DATA_DIR / 'phase_3_experiments')

x_test = leipzig_test[feature_cols].to_numpy(dtype=float)
y_test = leipzig_test['genus_latin'].map(label_to_idx).to_numpy()

# Scale with Berlin scaler (transfer learning)
x_test_scaled = berlin_scaler.transform(x_test)

print(f"\nLeipzig test: {len(leipzig_test)} samples")
print(f"Leipzig finetune: {len(leipzig_finetune)} samples")

In [ ]:
# Step 2: Zero-shot evaluation with bootstrap CIs
print(f"\n{'='*60}")
print(f"STEP 2: Zero-Shot Evaluation")
print(f"{'='*60}")

ml_preds = ml_model.predict(x_test_scaled)
ml_transfer_metrics = transfer.compute_transfer_metrics(
    y_test,
    ml_preds,
    class_labels=class_labels,
    include_ci=True,
    n_bootstrap=config['metrics']['n_bootstrap'],
)

print(f"\nML Zero-Shot Transfer:")
print(f"  F1: {ml_transfer_metrics['metrics']['f1_score']:.4f}")
print(f"  Accuracy: {ml_transfer_metrics['metrics']['accuracy']:.4f}")
if 'confidence_intervals' in ml_transfer_metrics:
    f1_ci = ml_transfer_metrics['confidence_intervals']['f1_score']
    print(f"  F1 95% CI: [{f1_ci[0]:.4f}, {f1_ci[1]:.4f}]")

In [ ]:
# Step 3: Transfer gap analysis
print(f"\n{'='*60}")
print(f"STEP 3: Transfer Gap Analysis")
print(f"{'='*60}")

# Load Berlin evaluation
berlin_eval = json.loads((OUTPUT_DIR / 'metadata/berlin_evaluation.json').read_text())
berlin_f1 = berlin_eval['metrics']['f1_score']
leipzig_f1 = ml_transfer_metrics['metrics']['f1_score']

gap = transfer.compute_transfer_gap(berlin_f1, leipzig_f1)

print(f"\nTransfer Gap:")
print(f"  Berlin F1: {berlin_f1:.4f}")
print(f"  Leipzig F1: {leipzig_f1:.4f}")
print(f"  Absolute drop: {gap.absolute_drop:.4f}")
print(f"  Relative drop: {gap.relative_drop:.1%}")
print(f"  Transfer efficiency: {gap.transfer_efficiency:.1%}")

In [ ]:
# Step 4: Leipzig from-scratch training
print(f"\n{'='*60}")
print(f"STEP 4: Leipzig From-Scratch Baseline")
print(f"{'='*60}")

# Fit Leipzig-specific scaler
x_finetune = leipzig_finetune[feature_cols].to_numpy(dtype=float)
y_finetune = leipzig_finetune['genus_latin'].map(label_to_idx).to_numpy()

leipzig_scaler = StandardScaler()
x_finetune_scaled = leipzig_scaler.fit_transform(x_finetune)
x_test_leipzig_scaled = leipzig_scaler.transform(x_test)

# Train from scratch with same hyperparameters
from_scratch_params = ml_metadata.get('best_params', {})
from_scratch_model = models.create_model(ml_name, model_params=from_scratch_params)
from_scratch_model.fit(x_finetune_scaled, y_finetune)

from_scratch_preds = from_scratch_model.predict(x_test_leipzig_scaled)
from_scratch_metrics = evaluation.compute_metrics(y_test, from_scratch_preds)

print(f"\nFrom-Scratch F1: {from_scratch_metrics['f1_score']:.4f}")
print(f"Transfer vs From-Scratch: {leipzig_f1 - from_scratch_metrics['f1_score']:+.4f}")

In [ ]:
# Step 5: Feature stability analysis
print(f"\n{'='*60}")
print(f"STEP 5: Feature Stability")
print(f"{'='*60}")

# Get Berlin importance (from Berlin model)
if hasattr(ml_model, 'feature_importances_'):
    berlin_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': ml_model.feature_importances_,
    })

# Get Leipzig importance (from from-scratch model)
if hasattr(from_scratch_model, 'feature_importances_'):
    leipzig_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': from_scratch_model.feature_importances_,
    })

    stability = transfer.compute_feature_stability(berlin_importance, leipzig_importance)
    
    print(f"\nFeature Stability:")
    print(f"  Spearman ρ: {stability['spearman_rho']:.3f}")
    print(f"  p-value: {stability['p_value']:.4f}")
    
    if stability['p_value'] < 0.05:
        print(f"  Significant correlation between Berlin and Leipzig feature rankings")
else:
    stability = None
    print(f"\nFeature importance not available for {ml_name}")

In [ ]:
# Step 6: Per-genus transfer robustness
print(f"\n{'='*60}")
print(f"STEP 6: Per-Genus Transfer Robustness")
print(f"{'='*60}")

# Per-genus metrics
berlin_per_genus = pd.DataFrame(berlin_eval['per_class'])
leipzig_per_genus = ml_transfer_metrics['per_class']

# Classify robustness
robustness = transfer.classify_transfer_robustness(
    berlin_per_genus,
    leipzig_per_genus,
    thresholds=config['transfer_evaluation']['robustness_thresholds'],
)

ranking = transfer.compute_transfer_robustness_ranking(robustness)

print(f"\nTransfer Robustness Summary:")
for category in ['robust', 'medium', 'poor']:
    genera = [g for g in ranking if robustness[g]['category'] == category]
    print(f"  {category.capitalize()}: {len(genera)} genera")

print(f"\nTop 5 most robust:")
for genus in ranking[:5]:
    print(f"  {genus}: F1 drop = {robustness[genus]['f1_drop']:.3f}")

print(f"\nTop 5 least robust:")
for genus in ranking[-5:]:
    print(f"  {genus}: F1 drop = {robustness[genus]['f1_drop']:.3f}")

In [ ]:
# Step 7: Hypothesis testing
print(f"\n{'='*60}")
print(f"STEP 7: A-Priori Hypotheses")
print(f"{'='*60}")

hypotheses = config['transfer_evaluation']['hypotheses']
hypothesis_results = []

# Prepare data for hypotheses
genus_data = pd.merge(
    berlin_per_genus[['genus', 'f1_score', 'support']].rename(columns={'f1_score': 'berlin_f1', 'support': 'berlin_n'}),
    leipzig_per_genus[['genus', 'f1_score']].rename(columns={'f1_score': 'leipzig_f1'}),
    on='genus',
)
genus_data['transfer_gap'] = genus_data['berlin_f1'] - genus_data['leipzig_f1']

# Test each hypothesis
for hyp in hypotheses:
    print(f"\n{hyp['id']}: {hyp['description']}")
    
    result = transfer.test_hypothesis(
        hyp,
        genus_data=genus_data,
        feature_importance=berlin_importance if stability else None,
    )
    
    hypothesis_results.append(result)
    
    if 'statistic' in result:
        print(f"  Statistic: {result['statistic']:.3f}")
    if 'p_value' in result:
        print(f"  p-value: {result['p_value']:.4f}")
    print(f"  Result: {result['conclusion']}")

In [ ]:
# Step 8: Visualization
print(f"\n{'='*60}")
print(f"STEP 8: Generate Figures")
print(f"{'='*60}")

fig_dir = OUTPUT_DIR / 'figures/transfer_evaluation'
fig_dir.mkdir(parents=True, exist_ok=True)

# Confusion matrix comparison
berlin_cm = np.array(berlin_eval['confusion_matrix'])
leipzig_cm = ml_transfer_metrics['confusion_matrix']

visualization.plot_confusion_comparison(
    berlin_cm,
    leipzig_cm,
    class_labels,
    output_path=fig_dir / 'confusion_comparison.png',
)

# Per-genus transfer
visualization.plot_per_genus_transfer(
    berlin_per_genus,
    leipzig_per_genus,
    output_path=fig_dir / 'per_genus_transfer.png',
)

# Step 9: Conifer/deciduous and species analysis on Leipzig (H3)
print(f"\n{'='*60}")
print(f"STEP 9: Leipzig-Specific Analysis")
print(f"{'='*60}")

# 9a: Conifer vs Deciduous on Leipzig
print(f"\nAnalyzing conifer vs deciduous on Leipzig...")
leipzig_cd_metrics = evaluation.analyze_conifer_deciduous(
    y_test,
    ml_preds,
    class_labels,
    config['genus_groups'],
)

print(f"  Leipzig Conifer F1: {leipzig_cd_metrics['conifer_f1']:.4f} (n={leipzig_cd_metrics['conifer_n']})")
print(f"  Leipzig Deciduous F1: {leipzig_cd_metrics['deciduous_f1']:.4f} (n={leipzig_cd_metrics['deciduous_n']})")

# Visualization
visualization.plot_transfer_conifer_deciduous(
    {
        genus: {
            'berlin_f1': next((r['f1_score'] for r in berlin_eval['per_class'] if r['genus'] == genus), 0),
            'leipzig_f1': next((r['f1_score'] for r in ml_transfer_metrics['per_class'].to_dict('records') if r['genus'] == genus), 0),
        }
        for genus in class_labels
    },
    set(config['genus_groups']['conifer']),
    output_path=fig_dir / 'transfer_conifer_deciduous.png',
)

# 9b: Species breakdown for low F1 genera on Leipzig
print(f"\nAnalyzing species breakdown on Leipzig...")
if 'species_latin' in leipzig_test.columns:
    leipzig_species_breakdown = evaluation.analyze_species_breakdown(
        y_test,
        ml_preds,
        class_labels,
        leipzig_test['species_latin'],
        genus_german_map=None,
        f1_threshold=0.50,
    )
    
    if leipzig_species_breakdown:
        print(f"  Found {len(leipzig_species_breakdown)} Leipzig genera with F1 < 0.50")
        for genus in list(leipzig_species_breakdown.keys())[:3]:
            print(f"    {genus}: {len(leipzig_species_breakdown[genus])} species")
    else:
        print(f"  All Leipzig genera have F1 >= 0.50")
else:
    print(f"  species_latin not available in Leipzig test data")
    leipzig_species_breakdown = None

# Step 10: NN champion evaluation + ML vs NN comparison (H3)
print(f"\n{'='*60}")
print(f"STEP 10: NN Champion Evaluation")
print(f"{'='*60}")

# Evaluate NN champion
print(f"\nEvaluating NN champion ({nn_name})...")
nn_preds = nn_model.predict(x_test_scaled)
nn_transfer_metrics = transfer.compute_transfer_metrics(
    y_test,
    nn_preds,
    class_labels=class_labels,
    include_ci=True,
    n_bootstrap=config['metrics']['n_bootstrap'],
)

print(f"\nNN Zero-Shot Transfer:")
print(f"  F1: {nn_transfer_metrics['metrics']['f1_score']:.4f}")
print(f"  Accuracy: {nn_transfer_metrics['metrics']['accuracy']:.4f}")

# ML vs NN comparison
print(f"\nML vs NN Comparison:")
print(f"  ML F1: {ml_transfer_metrics['metrics']['f1_score']:.4f}")
print(f"  NN F1: {nn_transfer_metrics['metrics']['f1_score']:.4f}")
print(f"  Difference: {nn_transfer_metrics['metrics']['f1_score'] - ml_transfer_metrics['metrics']['f1_score']:+.4f}")

# Select best model for transfer
if nn_transfer_metrics['metrics']['f1_score'] > ml_transfer_metrics['metrics']['f1_score']:
    best_transfer_model = 'nn'
    best_transfer_f1 = nn_transfer_metrics['metrics']['f1_score']
    print(f"\n  Best transfer model: NN ({nn_name})")
else:
    best_transfer_model = 'ml'
    best_transfer_f1 = ml_transfer_metrics['metrics']['f1_score']
    print(f"\n  Best transfer model: ML ({ml_name})")

print(f"\nFigures saved to {fig_dir}")

In [ ]:
# Save transfer evaluation results
transfer_eval_data = {
    'ml_champion': ml_name,
    'nn_champion': nn_name,
    'ml_zero_shot_metrics': ml_transfer_metrics['metrics'],
    'nn_zero_shot_metrics': nn_transfer_metrics['metrics'],
    'best_transfer_model': best_transfer_model,
    'best_transfer_f1': best_transfer_f1,
    'transfer_gap': {
        'berlin_f1': berlin_f1,
        'leipzig_f1': leipzig_f1,
        'absolute_drop': gap.absolute_drop,
        'relative_drop': gap.relative_drop,
        'transfer_efficiency': gap.transfer_efficiency,
    },
    'from_scratch_f1': from_scratch_metrics['f1_score'],
    'feature_stability': stability if stability else None,
    'per_genus_robustness': robustness,
    'robustness_ranking': ranking,
    'hypothesis_tests': hypothesis_results,
    'leipzig_conifer_deciduous': leipzig_cd_metrics,
    'leipzig_species_breakdown': {genus: len(df) for genus, df in leipzig_species_breakdown.items()} if leipzig_species_breakdown else None,
    'per_class': ml_transfer_metrics['per_class'].to_dict(orient='records'),
    'confusion_matrix': ml_transfer_metrics['confusion_matrix'].tolist(),
}

eval_path = OUTPUT_DIR / 'metadata/transfer_evaluation.json'
eval_path.write_text(json.dumps(transfer_eval_data, indent=2))

print(f"\nSaved {eval_path}")

In [ ]:
# Execution log
elapsed = time.time() - start_time

log = {
    'status': 'completed',
    'timestamp': datetime.datetime.now().isoformat(),
    'runtime_seconds': elapsed,
    'config_hash': hashlib.md5(json.dumps(config, sort_keys=True).encode()).hexdigest(),
    'transfer_gap': {
        'absolute_drop': gap.absolute_drop,
        'relative_drop': gap.relative_drop,
    },
    'robustness_summary': {
        'robust': len([g for g in ranking if robustness[g]['category'] == 'robust']),
        'medium': len([g for g in ranking if robustness[g]['category'] == 'medium']),
        'poor': len([g for g in ranking if robustness[g]['category'] == 'poor']),
    },
}

log_path = OUTPUT_DIR / 'logs/03c_transfer_evaluation.json'
log_path.write_text(json.dumps(log, indent=2))

print(f"\n{'='*60}")
print(f"TRANSFER EVALUATION COMPLETED")
print(f"{'='*60}")
print(f"Runtime: {elapsed/60:.1f} minutes")
print(f"Transfer gap: {gap.relative_drop:.1%}")
print(f"Output: {eval_path}")